In [6]:
# from pyspark.sql.session module import SparkSession class
from pyspark.sql.session import SparkSession

In [11]:
#  SparkSession.builder -- Creates a :class:`Builder` for constructing a :class:`SparkSession`

# Methods defined in SparkSession.builder:

# appName(self, name: str) -> 'SparkSession.Builder'
#  |      Sets a name for the application, which will be shown in the Spark web UI.
#  |      
#  |      If no application name is set, a randomly generated name will be used.

# config(self, key: Optional[str] = None, value: Optional[Any] = None, conf: Optional[pyspark.conf.SparkConf] = None, *, map: Optional[Dict[str, ForwardRef('OptionalPrimitiveType')]] = None) -> 'SparkSession.Builder'
#  |      Sets a config option. Options set using this method are automatically propagated to
#  |      both :class:`SparkConf` and :class:`SparkSession`'s own configuration.

#  getOrCreate(self) -> 'SparkSession'
#  |      Gets an existing :class:`SparkSession` or, if there is no existing one, creates a
#  |      new one based on the options set in this builder.
s1 = SparkSession.builder.config("k1", "v1").getOrCreate()
s1.conf.get("k1") == "v1"

True

In [12]:
# The configuration of the SparkSession can be changed afterwards
s1.conf.set("k1", "v1_new")
s1.conf.get("k1")

'v1_new'

In [14]:
# In case an existing SparkSession is returned, the config options specified
#  |      in this builder will be applied to the existing SparkSession.

s2 = SparkSession.builder.config("k2", "v2").getOrCreate()
s1.conf.get("k1") == s2.conf.get("k1") == "v1_new"

True

In [15]:
s1.conf.get("k2") == s2.conf.get("k2") == "v2"

True

In [18]:
# master(self, master: str) -> 'SparkSession.Builder'
#  |      Sets the Spark master URL to connect to, such as "local" to run locally, "local[4]"
#  |      to run locally with 4 cores, or "spark://master:7077" to run on a Spark standalone
#  |      cluster.

SparkSession.builder.master("locally")

In [ ]:
# remote(self, url: str) -> 'SparkSession.Builder'
#  |      Sets the Spark remote URL to connect to, such as "sc://host:port" to run
#  |      it via Spark Connect server.

# SparkSession.builder.remote("remote_url_goes_here")

In [7]:
help(SparkSession.builder)

Help on Builder in module pyspark.sql.session object:

class Builder(builtins.object)
 |  Builder() -> None
 |  
 |  Builder for :class:`SparkSession`.
 |  
 |  Methods defined here:
 |  
 |  __init__(self) -> None
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  appName(self, name: str) -> 'SparkSession.Builder'
 |      Sets a name for the application, which will be shown in the Spark web UI.
 |      
 |      If no application name is set, a randomly generated name will be used.
 |      
 |      .. versionadded:: 2.0.0
 |      
 |      .. versionchanged:: 3.4.0
 |          Supports Spark Connect.
 |      
 |      Parameters
 |      ----------
 |      name : str
 |          an application name
 |      
 |      Returns
 |      -------
 |      :class:`SparkSession.Builder`
 |      
 |      Examples
 |      --------
 |      >>> SparkSession.builder.appName("My app")
 |      <pyspark.sql.session.SparkSession.Builder...
 |  
 |  config(self, key: Optional[str]

In [24]:
# returns an instance of DataFrameReader
spark.read

In [30]:
# read csv files
# Write a DataFrame into a CSV file and read it back.
df = spark.createDataFrame([{"age": 100, "name": "shishir"}])
df.write.mode("overwrite").format("csv").save("/Users/shishir/Pyspark/SampleData/sample.csv")

# read the csv file as a DataFrame with 'nullValue' option set to 'shishir'
# To avoid going through the entire data once, disable ``inferSchema`` option or specify the schema explicitly using ``schema``.
spark.read.csv("/Users/shishir/Pyspark/SampleData/sample.csv", schema = df.schema, nullValue="shishir").show()

+---+----+
|age|name|
+---+----+
|100|NULL|
+---+----+



In [31]:
# format(self, source: str) -> 'DataFrameReader'
#  |      Specifies the input data source format.
# format could be eg: 'json', 'parquet', 'csv', 'text' etc
spark.read.format("csv")

In [36]:
# Write a DataFrame into a JSON file and read it back.
spark.createDataFrame([
        {"name": "shishir", "age": 24}]
    ).write.mode("overwrite").format("json").save("/Users/shishir/Pyspark/SampleData/sample.json")

# always try to provide the schema while reading data, to avoid spark to go thru the input data scanning process.
# Performing scans could be expensive if the input dataset size is huge. Providing schema explicilty avoids the scanning
spark.read.json("/Users/shishir/Pyspark/SampleData/sample.json", schema = df.schema).show()

spark.read.format('json').load("/Users/shishir/Pyspark/SampleData/sample.json").show()

+---+-------+
|age|   name|
+---+-------+
| 24|shishir|
+---+-------+

+---+-------+
|age|   name|
+---+-------+
| 24|shishir|
+---+-------+



In [ ]:
# spark.read.jdbc(url: , table: )
# Construct a :class:`DataFrame` representing the database table named ``table``
#  |      accessible via JDBC URL ``url`` and connection ``properties``.

In [49]:
# storing the files in a temporary directory and reading from the same temp directory
import tempfile
with tempfile.TemporaryDirectory() as d:
    # write the df in csv format, overwrite if the file already exists in the directory
    # Header option set to True means, write header as well
    spark.createDataFrame(
        [{"age":21, "name":"Shishir"}]
    ).write.option("header",True).mode("overwrite").format('csv').save(d)
    
    # read the csv file, include header, nullValue as Shishir from the temp directory
    spark.read.csv(d, header=True, schema = df.schema, nullValue="Shishir").show()
    
    spark.read.option("header",True).option("nullValue","Shishir").schema(df.schema).csv(d).show()
    
    # combine all the options inside the options()
    spark.read.options(header=True, nullValue="Shishir", schema=df.schema).csv(d).show()

+---+----+
|age|name|
+---+----+
| 21|NULL|
+---+----+

+---+----+
|age|name|
+---+----+
| 21|NULL|
+---+----+

+---+----+
|age|name|
+---+----+
| 21|NULL|
+---+----+



In [51]:
# Write a DataFrame into a ORC file and read it back.
import tempfile
with tempfile.TemporaryDirectory() as d:
    spark.createDataFrame(
        [{"age":21, "name":"Shishir"}]
    ).write.mode("overwrite").format('orc').save(d)
    
    spark.read.orc(d).show()


+---+-------+
|age|   name|
+---+-------+
| 21|Shishir|
+---+-------+



In [52]:
# Write a DataFrame into a Parquet file and read it back.
import tempfile
with tempfile.TemporaryDirectory() as d:
    spark.createDataFrame(
        [{"age":21, "name":"Shishir"}]
    ).write.mode("overwrite").format('parquet').save(d)
    
    spark.read.parquet(d).show()

+---+-------+
|age|   name|
+---+-------+
| 21|Shishir|
+---+-------+



In [56]:
# table(self, tableName: str) -> 'DataFrame'
#  |      Returns the specified table as a :class:`DataFrame`.
spark.range(5).createOrReplaceTempView('table1')
spark.read.table("table1").show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [82]:
# Write a DataFrame into a text file and read it back.

import tempfile
with tempfile.TemporaryDirectory() as d:
    df = spark.createDataFrame(
        data = [("a",), ("b",), ("c",)], 
        schema = ["alphabets"]
    )
    df.write.mode("overwrite").format('text').save(d)
    
    # enforce schema while reading. Default columnname while reading text file is value. Since we enforced schema, the column name changed.
    # sort the df based on 'alphabets' column
    spark.read.schema(df.schema).text(d).sort("alphabets").show()
        

+---------+
|alphabets|
+---------+
|        a|
|        b|
|        c|
+---------+



In [19]:
help(spark.read)

Help on DataFrameReader in module pyspark.sql.readwriter object:

class DataFrameReader(OptionUtils)
 |  DataFrameReader(spark: 'SparkSession')
 |  
 |  Interface used to load a :class:`DataFrame` from external storage systems
 |  (e.g. file systems, key-value stores, etc). Use :attr:`SparkSession.read`
 |  to access this.
 |  
 |  .. versionadded:: 1.4.0
 |  
 |  .. versionchanged:: 3.4.0
 |      Supports Spark Connect.
 |  
 |  Method resolution order:
 |      DataFrameReader
 |      OptionUtils
 |      builtins.object
 |  
 |  Methods defined here:
 |  
 |  __init__(self, spark: 'SparkSession')
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  csv(self, path: Union[str, List[str]], schema: Union[pyspark.sql.types.StructType, str, NoneType] = None, sep: Optional[str] = None, encoding: Optional[str] = None, quote: Optional[str] = None, escape: Optional[str] = None, comment: Optional[str] = None, header: Union[bool, str, NoneType] = None, inferSchema: Unio

In [53]:
help(spark.table)

Help on method table in module pyspark.sql.session:

table(tableName: str) -> pyspark.sql.dataframe.DataFrame method of pyspark.sql.session.SparkSession instance
    Returns the specified table as a :class:`DataFrame`.
    
    .. versionadded:: 2.0.0
    
    .. versionchanged:: 3.4.0
        Supports Spark Connect.
    
    Parameters
    ----------
    tableName : str
        the table name to retrieve.
    
    Returns
    -------
    :class:`DataFrame`
    
    Examples
    --------
    >>> spark.range(5).createOrReplaceTempView("table1")
    >>> spark.table("table1").sort("id").show()
    +---+
    | id|
    +---+
    |  0|
    |  1|
    |  2|
    |  3|
    |  4|
    +---+



In [63]:
type(["a"])

list